# Head Pruning — Iterative at 25 / 50 / 75 %

Scores all 144 attention heads by gradient magnitude, then iteratively removes
the least important ones, retrains 5 epochs from the previous checkpoint,
and compares Attention Rollout maps across sparsity levels.

**Runs on Colab (A100). MacBook: set IS_COLAB=False — uses dev5k, 1 epoch.**

In [ ]:
# ── Cell 1 — Environment ──────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/dizertatie_project'):
        !git clone https://github.com/daviDpaD18/diz.git /content/diz
        os.symlink('/content/diz/dizertatie_project', '/content/dizertatie_project')
    sys.path.insert(0, '/content/dizertatie_project')
except ImportError:
    IS_COLAB = False
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

print('Colab:', IS_COLAB)

In [ ]:
# ── Cell 2 — Data to local SSD (Colab only) ───────────────────────────────────
if IS_COLAB:
    DRIVE = '/content/drive/MyDrive/dizertatie'
    if not os.path.exists('/content/train'):
        print('Copying train.zip ...')
        !cp {DRIVE}/train.zip /content/train.zip
        !unzip -q /content/train.zip -d /content/
        !rm /content/train.zip
        print('train/ ready.')
    if not os.path.exists('/content/valid'):
        !cp -r {DRIVE}/valid /content/valid
        print('valid/ ready.')
    import importlib, src.config
    importlib.reload(src.config)

import json, numpy as np, pandas as pd
import torch, torch.nn.functional as F, timm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score

from src.config import IMAGE_ROOT, SPLITS_DIR, WEIGHTS_DIR, CKPT_DIR
from src.dataset import LABEL_COLS, CheXpertDataset, train_transforms, eval_transforms

CKPT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps')  if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print('Device:', DEVICE)

In [ ]:
# ── Cell 3 — Constants, model loading, class weights ──────────────────────────
NUM_BLOCKS  = 12
NUM_HEADS   = 12
HEAD_DIM    = 768 // NUM_HEADS   # 64
TOTAL_HEADS = NUM_BLOCKS * NUM_HEADS   # 144

SPARSITY_LEVELS = [0.25, 0.50, 0.75]
RETRAIN_EPOCHS  = 5 if IS_COLAB else 1
BATCH_SIZE      = 32 if IS_COLAB else 8
NUM_WORKERS     = 4  if IS_COLAB else 0
LR              = 5e-5
WEIGHT_DECAY    = 1e-2
GRAD_CLIP       = 1.0

# Pick best seed by val AUC
seeds = [42, 123, 456]
best_seed, best_auc = max(
    ((s, torch.load(CKPT_DIR / f'baseline_seed{s}_best.pt',
                    map_location='cpu', weights_only=False)['val_macro_auc'])
     for s in seeds),
    key=lambda x: x[1]
)
BASELINE_CKPT = CKPT_DIR / f'baseline_seed{best_seed}_best.pt'
print(f'Best baseline: seed {best_seed}, AUC {best_auc:.4f}')
print(f'Total heads: {TOTAL_HEADS}  |  Removed at each level: '
      + ', '.join(f'{int(s*TOTAL_HEADS)}' for s in SPARSITY_LEVELS))

# Always read architecture from checkpoint — never hardcode
def load_model(ckpt_path):
    ckpt       = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model_name = ckpt['hparams']['model']
    model      = timm.create_model(model_name, pretrained=False, num_classes=14)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.to(DEVICE), ckpt

model, baseline_ckpt = load_model(BASELINE_CKPT)
print(f'Architecture: {baseline_ckpt["hparams"]["model"]}')

# Class weights — identical to baseline training
with open(WEIGHTS_DIR / 'class_weights.json') as f:
    cw = json.load(f)
CLASS_WEIGHTS = torch.tensor([cw[l] for l in LABEL_COLS], dtype=torch.float32).to(DEVICE)

def weighted_bce(logits, labels):
    return (F.binary_cross_entropy_with_logits(
        logits, labels, reduction='none') * CLASS_WEIGHTS).mean()
# Output directories — point to Drive on Colab so everything persists
MAPS_DIR = (CKPT_DIR.parent / 'maps') if IS_COLAB \
           else __import__('pathlib').Path('..').resolve() / 'maps'
(MAPS_DIR / 'gradcam').mkdir(parents=True, exist_ok=True)
(MAPS_DIR / 'rollout').mkdir(parents=True, exist_ok=True)
print('MAPS_DIR:', MAPS_DIR)


In [ ]:
# ── Cell 4 — Data loaders ─────────────────────────────────────────────────────
TRAIN_CSV = SPLITS_DIR / ('train_full.csv' if IS_COLAB else 'train_dev5k.csv')
VALID_CSV = SPLITS_DIR / 'valid_frontal.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)

train_ds     = CheXpertDataset(df_train, IMAGE_ROOT, train_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB, drop_last=True)
val_ds       = CheXpertDataset(df_valid, IMAGE_ROOT, eval_transforms)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB)

print(f'Train: {len(df_train):,}  |  Val: {len(df_valid):,}')

In [ ]:
# ── Cell 5 — Head importance scoring ─────────────────────────────────────────
#
# For each head h in block l, importance = mean absolute gradient of the loss
# w.r.t. the output-projection weight columns that belong to that head.
#
# The output projection weight is [768, 768]. Columns [h*64 : (h+1)*64]
# are the weights that receive head h's output. Their gradient tells us
# how much removing head h's contribution would change the loss.
# Accumulated over all batches and normalised.

scores = torch.zeros(NUM_BLOCKS, NUM_HEADS)

model.train()
for imgs, labels, *_ in tqdm(train_loader, desc='Scoring heads'):
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    model.zero_grad()
    weighted_bce(model(imgs), labels).backward()

    for l, block in enumerate(model.blocks):
        grad = block.attn.proj.weight.grad   # [768, 768]
        for h in range(NUM_HEADS):
            s = h * HEAD_DIM
            scores[l, h] += grad[:, s:s + HEAD_DIM].abs().sum().item()

scores /= len(train_loader)
model.eval()
print(f'Scoring done. Range: [{scores.min():.4f}, {scores.max():.4f}]')

In [ ]:
# ── Cell 6 — Visualise importance scores ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

im = ax1.imshow(scores.numpy(), aspect='auto', cmap='viridis')
plt.colorbar(im, ax=ax1, label='Importance')
ax1.set_xlabel('Head'); ax1.set_ylabel('Block')
ax1.set_title('Head importance heatmap (rows=blocks, cols=heads)')
ax1.set_xticks(range(NUM_HEADS)); ax1.set_yticks(range(NUM_BLOCKS))

flat_sorted = scores.flatten().sort().values.numpy()
ax2.plot(flat_sorted, color='steelblue')
for s in SPARSITY_LEVELS:
    idx = int(s * TOTAL_HEADS)
    ax2.axvline(idx, linestyle='--', label=f'{int(s*100)}% ({idx} heads)')
    ax2.axhline(flat_sorted[idx-1], linestyle=':', alpha=0.4)
ax2.set_xlabel('Head rank (least → most important)')
ax2.set_ylabel('Score'); ax2.set_title('Sorted importance scores')
ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 7 — Head mask utilities ─────────────────────────────────────────────
#
# get_head_mask: returns [12, 12] bool tensor — False means the head is pruned.
#
# apply_head_mask: registers a forward pre-hook on each block's attn.proj.
# The hook receives the concatenated head outputs [B, N, 768] and zeros the
# 64-dim slice belonging to each pruned head before the projection runs.
# This is reversible — hooks are removed after each retraining round.

def get_head_mask(scores: torch.Tensor, sparsity: float) -> torch.Tensor:
    order = scores.flatten().argsort()           # ascending: worst first
    n_remove = int(sparsity * TOTAL_HEADS)
    mask = torch.ones(TOTAL_HEADS, dtype=torch.bool)
    mask[order[:n_remove]] = False
    return mask.reshape(NUM_BLOCKS, NUM_HEADS)


def apply_head_mask(model, head_mask: torch.Tensor):
    handles = []
    for l, block in enumerate(model.blocks):
        row = head_mask[l].float().to(DEVICE)    # [12]
        def make_hook(m):
            def hook(module, args):
                x = args[0]                      # [B, N, 768]
                B, N, _ = x.shape
                x = x.reshape(B, N, NUM_HEADS, HEAD_DIM)
                x = x * m.view(1, 1, -1, 1)
                return (x.reshape(B, N, -1),)
            return hook
        handles.append(block.attn.proj.register_forward_pre_hook(make_hook(row)))
    return handles


def remove_hooks(handles):
    for h in handles: h.remove()


for s in SPARSITY_LEVELS:
    m = get_head_mask(scores, s)
    print(f'{int(s*100)}%  — heads kept: {m.sum().item()} / {TOTAL_HEADS}')

In [ ]:
# ── Cell 8 — Validation function ─────────────────────────────────────────────
@torch.no_grad()
def evaluate(model):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, *_ in val_loader:
        all_probs.append(torch.sigmoid(model(imgs.to(DEVICE))).cpu())
        all_labels.append(labels)
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = {}
    for i, label in enumerate(LABEL_COLS):
        auc[label] = roc_auc_score(labels[:, i], probs[:, i]) \
                     if labels[:, i].sum() > 0 else float('nan')
    return float(np.nanmean(list(auc.values()))), auc

In [ ]:
# ── Cell 9 — Iterative pruning + retraining ───────────────────────────────────
#
# 25%: load baseline    → apply 25% mask → retrain 5 epochs → save
# 50%: load 25% ckpt    → apply 50% mask → retrain 5 epochs → save
# 75%: load 50% ckpt    → apply 75% mask → retrain 5 epochs → save
#
# Starting from the previous checkpoint (not always baseline) is the
# 'iterative' part — the model has already adapted to the previous
# pruning level before the next heads are removed.
# The importance ranking comes from the original baseline scores throughout.

pruning_results = {}
prev_ckpt       = BASELINE_CKPT

for sparsity in SPARSITY_LEVELS:
    pct = int(sparsity * 100)
    print(f'\n{"="*60}\nPRUNING {pct}%  ({int(sparsity*TOTAL_HEADS)} heads removed)\n{"="*60}')

    model, _ = load_model(prev_ckpt)
    head_mask = get_head_mask(scores, sparsity)
    handles   = apply_head_mask(model, head_mask)

    macro_pre, _ = evaluate(model)
    print(f'AUC immediately after masking (before retrain): {macro_pre:.4f}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = RETRAIN_EPOCHS * len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps)
    scaler    = torch.amp.GradScaler() if torch.cuda.is_available() else None

    best_auc_pruned = 0.0
    save_path = CKPT_DIR / f'pruned_{pct}pct_best.pt'

    for epoch in range(1, RETRAIN_EPOCHS + 1):
        model.train()
        running = 0.0
        for imgs, labels, *_ in tqdm(train_loader, desc=f'Epoch {epoch}/{RETRAIN_EPOCHS}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    loss = weighted_bce(model(imgs), labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            else:
                loss = weighted_bce(model(imgs), labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            scheduler.step()
            running += loss.item()

        macro, auc_per_label = evaluate(model)
        flag = ' ← best' if macro > best_auc_pruned else ''
        print(f'  Epoch {epoch} | loss {running/len(train_loader):.4f} | AUC {macro:.4f}{flag}')

        if macro > best_auc_pruned:
            best_auc_pruned = macro
            torch.save({
                'sparsity': sparsity, 'head_mask': head_mask,
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'val_macro_auc': macro, 'val_auc_per_label': auc_per_label,
                'auc_before_retrain': macro_pre,
                'hparams': baseline_ckpt['hparams'],
            }, save_path)

    remove_hooks(handles)
    pruning_results[pct] = {'auc_before': macro_pre, 'auc_after': best_auc_pruned}
    prev_ckpt = save_path
    print(f'Saved {save_path.name}  |  {macro_pre:.4f} → {best_auc_pruned:.4f}')

In [ ]:
# ── Cell 10 — Pruning summary table ──────────────────────────────────────────
print(f'\n{"="*58}')
print('HEAD PRUNING RESULTS')
print(f'{"="*58}')
print(f'{"Sparsity":<10} {"Kept":<8} {"Pre-retrain":<14} {"Post-retrain":<14} {"vs baseline"}')
print('-' * 58)
print(f'{"Baseline":<10} {144:<8} {"-":<14} {best_auc:<14.4f} {"-"}')
for pct, r in pruning_results.items():
    kept  = TOTAL_HEADS - int(pct * TOTAL_HEADS / 100)
    delta = r['auc_after'] - best_auc
    print(f'{pct}%{"":<8} {kept:<8} {r["auc_before"]:<14.4f} {r["auc_after"]:<14.4f} {delta:+.4f}')

In [ ]:
# ── Cell 11 — Per-label AUC comparison ───────────────────────────────────────
# Loads saved auc_per_label from each checkpoint and shows a full table:
# baseline vs 25% vs 50% vs 75% for every label.
# Highlights which pathologies are most sensitive to pruning.

import warnings

RELIABLE_MIN = 5   # min val positives to report AUC
df_valid_check = pd.read_csv(VALID_CSV)
val_pos = df_valid_check[LABEL_COLS].sum().to_dict()

ckpt_names = ['baseline'] + [f'{int(s*100)}pct' for s in SPARSITY_LEVELS]
ckpt_paths = [BASELINE_CKPT] + \
             [CKPT_DIR / f'pruned_{int(s*100)}pct_best.pt' for s in SPARSITY_LEVELS]

auc_table = {}   # {label: {ckpt_name: auc}}
for name, path in zip(ckpt_names, ckpt_paths):
    ckpt_data = torch.load(path, map_location='cpu', weights_only=False)
    for label, val in ckpt_data['val_auc_per_label'].items():
        auc_table.setdefault(label, {})[name] = val

rows = []
for label in LABEL_COLS:
    n_pos    = int(val_pos.get(label, 0))
    reliable = '✓' if n_pos >= RELIABLE_MIN else f'✗ (n={n_pos})'
    row = {'Label': label, 'Val+': n_pos, 'Reliable': reliable}
    for name in ckpt_names:
        row[name] = auc_table.get(label, {}).get(name, float('nan'))
    # Delta: pruned vs baseline
    base_auc = row['baseline']
    for name in ckpt_names[1:]:
        row[f'Δ{name}'] = row[name] - base_auc if not (isinstance(row[name], float) and\
                           row[name] != row[name]) else float('nan')
    rows.append(row)

df_auc = pd.DataFrame(rows).set_index('Label')
cols_show = ['Val+', 'Reliable', 'baseline'] + \
            [f'{int(s*100)}pct' for s in SPARSITY_LEVELS] + \
            [f'Δ{int(s*100)}pct' for s in SPARSITY_LEVELS]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(df_auc[cols_show].round(4).to_string())

# Macro over reliable labels
for name in ckpt_names:
    vals = [auc_table.get(l, {}).get(name, float('nan'))
            for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        macro = float(np.nanmean(vals))
    print(f'Macro AUC (reliable, {name}): {macro:.4f}')

df_auc[cols_show].to_csv(CKPT_DIR / 'pruning_auc_comparison.csv')
print('\nSaved pruning_auc_comparison.csv')

In [ ]:
# ── Cell 11 — Attention Rollout utility ───────────────────────────────────────
#
# Identical to notebook 03 but accepts head_mask so pruned heads are zeroed
# in the rollout computation — they shouldn't route information if their
# output is already masked.

def compute_rollout_batch(model, imgs, head_mask=None):
    for block in model.blocks: block.attn.fused_attn = False
    attn_w = {}
    hooks  = []
    def make_hook(i):
        def h(m, inp, out): attn_w[i] = out.detach().cpu()
        return h
    for i, block in enumerate(model.blocks):
        hooks.append(block.attn.attn_drop.register_forward_hook(make_hook(i)))
    with torch.no_grad(): _ = model(imgs)
    for h in hooks: h.remove()
    for block in model.blocks: block.attn.fused_attn = True

    B, N = imgs.shape[0], 197
    I = torch.eye(N)
    rollout = I.unsqueeze(0).expand(B, -1, -1).clone()
    for i in range(NUM_BLOCKS):
        attn = attn_w[i]   # [B, H, N, N]
        if head_mask is not None:
            attn = attn * head_mask[i].float().view(1, NUM_HEADS, 1, 1)
        attn = attn.mean(1)
        attn = 0.5 * attn + 0.5 * I
        attn = attn / attn.sum(-1, keepdim=True)
        rollout = attn @ rollout
    return rollout[:, 0, 1:].reshape(B, 14, 14).numpy()

In [ ]:
# ── Cell 12 — Generate rollout maps for all checkpoints ───────────────────────

val_loader_rb = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

to_eval = [('baseline', BASELINE_CKPT)] + \
          [(f'{int(s*100)}pct', CKPT_DIR / f'pruned_{int(s*100)}pct_best.pt')
           for s in SPARSITY_LEVELS]

rollout_collection = {}
for name, path in to_eval:
    ckpt_data = torch.load(path, map_location='cpu', weights_only=False)
    mdl, _    = load_model(path)
    hm        = ckpt_data.get('head_mask', None)
    hdls      = apply_head_mask(mdl, hm) if hm is not None else []

    maps = [compute_rollout_batch(mdl, imgs.to(DEVICE), hm)
            for imgs, *_ in tqdm(val_loader_rb, desc=f'Rollout [{name}]')]
    remove_hooks(hdls)

    arr = np.concatenate(maps, axis=0)   # [202, 14, 14]
    rollout_collection[name] = arr
    np.save(MAPS_DIR / 'rollout' / f'rollout_{name}.npy', arr)
    print(f'  {name}: saved {arr.shape}')

In [ ]:
# ── Cell 13 — Rollout comparison ─────────────────────────────────────────────
#
# Cosine similarity between baseline rollout and pruned rollout per val image.
# Mean near 1.0 = pruning preserved attention routing.
# Mean dropping with sparsity = information pathways are being disrupted.

base = rollout_collection['baseline'].reshape(len(df_valid), -1)   # [202, 196]

print(f'{"Sparsity":<12} {"Mean cos-sim":<16} Std')
print('-' * 38)
sims = {}
for s in SPARSITY_LEVELS:
    name   = f'{int(s*100)}pct'
    pruned = rollout_collection[name].reshape(len(df_valid), -1)
    dot    = (base * pruned).sum(1)
    cos    = dot / (np.linalg.norm(base, axis=1) * np.linalg.norm(pruned, axis=1) + 1e-8)
    sims[name] = cos
    print(f'{int(s*100)}%{"":<10} {cos.mean():.4f}{"":<12} {cos.std():.4f}')

# Side-by-side visualisation for one sample image
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (n, t) in zip(axes, [('baseline','Baseline'),('25pct','25%'),('50pct','50%'),('75pct','75%')]):
    m = rollout_collection[n][0]
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    ax.imshow(m, cmap='hot', interpolation='bilinear')
    ax.set_title(t); ax.axis('off')
plt.suptitle('Attention Rollout — val image 0 across sparsity levels')
plt.tight_layout()
plt.savefig(MAPS_DIR / 'rollout' / 'rollout_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# grad-cam is the PyPI name — import is still 'pytorch_grad_cam'
!pip install -q grad-cam

# ── Cell 12 — Grad-CAM generation for all checkpoints ─────────────────────────
# Generates 14×14 Grad-CAM maps for all 202 val images × 14 labels
# for the baseline and each pruned checkpoint.
# Uses the same reshape_transform as notebook 03.
# Saved as [202, 14, 14, 14] npy arrays (images × labels × h × w).

import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

(MAPS_DIR / 'gradcam').mkdir(parents=True, exist_ok=True)

def reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, -1)
    return result.permute(0, 3, 1, 2)   # [B, 768, 14, 14]

val_loader_cam = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
N_VAL         = len(df_valid)
gradcam_collection = {}

for name, path in zip(ckpt_names, ckpt_paths):
    ckpt_data = torch.load(path, map_location='cpu', weights_only=False)
    mdl, _    = load_model(path)
    hm        = ckpt_data.get('head_mask', None)
    hdls      = apply_head_mask(mdl, hm) if hm is not None else []

    cam = GradCAM(
        model=mdl,
        target_layers=[mdl.blocks[-1].norm1],
        reshape_transform=reshape_transform,
    )

    maps = np.zeros((N_VAL, len(LABEL_COLS), 14, 14), dtype=np.float32)
    for img_idx, (imgs, *_) in enumerate(tqdm(val_loader_cam,
                                               desc=f'Grad-CAM [{name}]')):
        imgs = imgs.to(DEVICE)
        for li in range(len(LABEL_COLS)):
            cam_224 = cam(input_tensor=imgs,
                          targets=[ClassifierOutputTarget(li)])
            maps[img_idx, li] = cv2.resize(cam_224[0], (14, 14),
                                            interpolation=cv2.INTER_AREA)

    remove_hooks(hdls)
    gradcam_collection[name] = maps
    save_p = MAPS_DIR / 'gradcam' / f'gradcam_{name}.npy'
    np.save(save_p, maps)
    print(f'  {name}: saved {maps.shape}')

In [ ]:
# ── Cell 13 — Grad-CAM map similarity ────────────────────────────────────────
# Cosine similarity between baseline Grad-CAM and pruned Grad-CAM per label.
# Only computed over images that are positive for the label (Grad-CAM on a
# negative image is not interpretable).
#
# Research question: are class-discriminative regions (Grad-CAM) more stable
# under pruning than the overall routing maps (Attention Rollout)?

base_gcam = gradcam_collection['baseline']   # [202, 14, 14, 14]
is_pos    = df_valid_check[LABEL_COLS].values == 1.0   # [202, 14]

print('Grad-CAM cosine similarity vs baseline (positive images only)')
print(f'{"":<35} {" ".join(f"{int(s*100)}%"[:5].ljust(8) for s in SPARSITY_LEVELS)}')
print('-' * 60)

gcam_sim_results = {}   # {label: {pct: mean_cos_sim}}

for li, label in enumerate(LABEL_COLS):
    pos_idx = np.where(is_pos[:, li])[0]
    if len(pos_idx) < RELIABLE_MIN:
        continue

    base_flat = base_gcam[pos_idx, li].reshape(len(pos_idx), -1)   # [N+, 196]
    row_str   = f'{label:<35}'
    gcam_sim_results[label] = {}

    for s in SPARSITY_LEVELS:
        name  = f'{int(s*100)}pct'
        prn   = gradcam_collection[name][pos_idx, li].reshape(len(pos_idx), -1)
        dot   = (base_flat * prn).sum(1)
        cos   = dot / (np.linalg.norm(base_flat, axis=1) *
                       np.linalg.norm(prn, axis=1) + 1e-8)
        m = cos.mean()
        gcam_sim_results[label][int(s*100)] = m
        row_str += f'{m:.4f}   '
    print(row_str)

# Mean across all reliable labels
print('\nMean across reliable labels:')
row_str = f'{"":<35}'
for s in SPARSITY_LEVELS:
    pct  = int(s * 100)
    vals = [gcam_sim_results[l][pct] for l in gcam_sim_results if pct in gcam_sim_results[l]]
    row_str += f'{np.mean(vals):.4f}   '
print(row_str)

In [ ]:
# ── Cell 14 — Combined comparison: Rollout vs Grad-CAM stability ──────────────
# Side-by-side plot showing mean cosine similarity for both map types
# as sparsity increases.
# If Grad-CAM sim stays high while Rollout sim drops → information is
# re-routed but diagnostic focus is preserved (supports the thesis argument).

sparsity_pcts = [int(s * 100) for s in SPARSITY_LEVELS]

# Rollout similarities from Cell 10 (sims dict)
rollout_means = [sims[f'{p}pct'].mean() for p in sparsity_pcts]

# Grad-CAM similarities — mean over all reliable labels
gcam_means = []
for p in sparsity_pcts:
    vals = [gcam_sim_results[l][p] for l in gcam_sim_results if p in gcam_sim_results[l]]
    gcam_means.append(np.mean(vals) if vals else float('nan'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: similarity curves
axes[0].plot(sparsity_pcts, rollout_means, 'o-', label='Attention Rollout', color='steelblue')
axes[0].plot(sparsity_pcts, gcam_means,    's-', label='Grad-CAM',          color='coral')
axes[0].set_xlabel('Sparsity (%)')
axes[0].set_ylabel('Mean cosine similarity vs baseline')
axes[0].set_title('Map stability under head pruning')
axes[0].set_ylim(0, 1.05)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: AUC degradation
auc_post  = [pruning_results[p]['auc_after']  for p in sparsity_pcts]
auc_pre   = [pruning_results[p]['auc_before'] for p in sparsity_pcts]
axes[1].axhline(best_auc, linestyle='--', color='gray', label='Baseline AUC')
axes[1].plot(sparsity_pcts, auc_pre,  'v--', label='Post-mask (pre-retrain)', color='orange', alpha=0.7)
axes[1].plot(sparsity_pcts, auc_post, 'o-',  label='Post-retrain',            color='green')
axes[1].set_xlabel('Sparsity (%)')
axes[1].set_ylabel('Macro AUC (reliable labels)')
axes[1].set_title('AUC vs sparsity')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MAPS_DIR / 'head_pruning_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {MAPS_DIR / "head_pruning_comparison.png"}')

In [ ]:
# ── Cell 15 — Visual overlay grid ────────────────────────────────────────────
# For a few pathologies, show Grad-CAM and Rollout maps side by side
# for baseline and each pruned level on the same image.
# Rows = pathologies, columns = baseline / 25% / 50% / 75%.

from PIL import Image as PILImage

SHOW_LABELS = ['Pleural Effusion', 'Cardiomegaly', 'Edema']

def overlay_cam(img_path, cam_14, alpha=0.5):
    orig = np.array(PILImage.open(img_path).convert('RGB'))
    h, w = orig.shape[:2]
    hm   = cv2.resize(cam_14.astype(np.float32), (w, h))
    hm   = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
    hm   = cv2.applyColorMap((hm * 255).astype(np.uint8), cv2.COLORMAP_JET)
    hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    return (alpha * hm + (1 - alpha) * orig).astype(np.uint8)

fig, axes = plt.subplots(
    len(SHOW_LABELS), 4 * 2,   # 4 sparsity levels × (grad-cam + rollout)
    figsize=(22, 4 * len(SHOW_LABELS))
)
levels = [('baseline', 'Baseline'), ('25pct', '25%'), ('50pct', '50%'), ('75pct', '75%')]

for row, label in enumerate(SHOW_LABELS):
    li      = LABEL_COLS.index(label)
    pos_idx = df_valid_check.index[df_valid_check[label] == 1.0].tolist()
    if not pos_idx:
        continue
    img_idx  = pos_idx[0]
    img_path = IMAGE_ROOT / df_valid_check.iloc[img_idx]['Path']
    orig     = np.array(PILImage.open(img_path).convert('RGB'))
    h, w     = orig.shape[:2]

    for col, (name, title) in enumerate(levels):
        # Grad-CAM overlay
        ax_g = axes[row, col * 2]
        ax_g.imshow(overlay_cam(img_path, gradcam_collection[name][img_idx, li]))
        ax_g.set_title(f'{title}\nGrad-CAM ({label})', fontsize=7)
        ax_g.axis('off')

        # Rollout
        ax_r = axes[row, col * 2 + 1]
        rm   = rollout_collection[name][img_idx]
        rm   = cv2.resize(rm, (w, h))
        rm   = (rm - rm.min()) / (rm.max() - rm.min() + 1e-8)
        ax_r.imshow(rm, cmap='hot')
        ax_r.set_title(f'{title}\nRollout', fontsize=7)
        ax_r.axis('off')

plt.suptitle('Grad-CAM and Attention Rollout across pruning levels', y=1.01)
plt.tight_layout()
plt.savefig(MAPS_DIR / 'head_pruning_visual_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {MAPS_DIR / "head_pruning_visual_grid.png"}')